# Matched Pythia continuation

Runs baseline and CPS-prescribed forks from identical model weights and identical subsequent batches.

In [ ]:
import os, pathlib, subprocess, sys
REPO_URL=os.environ.get("CPS_REPO_URL","https://github.com/fyremael/CPS.git")
GIT_REF=os.environ.get("CPS_GIT_REF","main")
repo=pathlib.Path("/content/CPS")
if not repo.exists(): subprocess.run(["git","clone","--depth","1","--branch",GIT_REF,REPO_URL,str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",".[pythia,notebooks]"],check=True)

In [ ]:
import os
from cps.pythia.continuation import ContinuationConfig, ContinuationControl, run_matched_continuation
config=ContinuationConfig(
    revision=os.environ.get("CPS_REVISION","step1000"),
    steps=int(os.environ.get("CPS_CONTINUATION_STEPS","20")),
    intervention=ContinuationControl(
        name="cps",
        learning_rate_scale=float(os.environ.get("CPS_LR_SCALE","0.8")),
        beta1=float(os.environ["CPS_BETA1"]) if "CPS_BETA1" in os.environ else None,
    ),
    output_dir="/content/cps-artifacts/continuation",
)
result=run_matched_continuation(config)
print(result)

In [ ]:
import pathlib, shutil
export=pathlib.Path("/content/cps-export"); export.mkdir(exist_ok=True)
shutil.copytree("/content/cps-artifacts",export/"artifacts",dirs_exist_ok=True)
shutil.make_archive("/content/cps-export","zip",export)